In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.hparam_searcher import HParamSearcher


In [2]:
# ── 실험 횟수 ────────────────────────────────────────────────────────────────
N_TRIALS = 10000

# ── 고정 학습 설정 ────────────────────────────────────────────────────────────
N_FOLDS        = 4
MAX_EPOCHS     = 1000
EARLY_STOPPING = True
PATIENCE       = 40
LOSS           = 'weighted_bce'

# ── 탐색 공간 ─────────────────────────────────────────────────────────────────
PARAM_GRID: dict[str, list] = {
    'random_state':          [42, 0, 1, 7, 13, 21, 99, 123, 777, 2024],
    'imputation_strategy':   ['knn', 'mice', 'mean', 'median', 'most_frequent'],
    'mice_max_iter':         [5, 10, 15, 20, 30],
    'knn_n_neighbors':       [5, 10, 20, 30, 40, 50],
    'imputation_fill_value': [0.0, -1.0],
    'hidden_layer_sizes':    [
        (64,), (128,), (256,),
        (64, 128), (128, 64), (128, 256), (256, 128),
        (64, 128, 256), (256, 128, 64), (128, 64, 32),
        (256, 128, 64, 32),
    ],
    'dropout_rate':          [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    'learning_rate':         [1e-4, 3e-4, 5e-4, 1e-3, 3e-3, 5e-3],
    'weight_decay':          [0.0, 1e-5, 1e-4, 1e-3, 1e-2],
    'batch_size':            [8, 16, 32, 64, 128],
    'validation_fraction':   [0.1, 0.15, 0.2, 0.25, 0.3],
}

# ── 피처 목록 ─────────────────────────────────────────────────────────────────
DIABETES_FEATURES: list[str] | None = ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'Ca', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'K', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Na', 'Neutroph', 'P', 'PCT', 'PDW', 'PH', 'PSA', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride', 'UIBC', 'Uric acid', 'WBC', 'current_glucose', 'e-GFR', 'future_glucose', 'glucose_change', 'r-GTP', '공복혈당', '나이', '당화혈색소', '맥박', '비만도', '시력(우안)', '시력(좌안)', '신장', '안압(우안)', '안압(좌안)', '철포화율', '청력우(1000Hz)', '청력좌(1000Hz)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']
PREDIABETES_FEATURES: list[str] | None = ['A/G ratio', 'AFP', 'ALP', 'Albumin', 'B/C ratio', 'BMI', 'BUN', 'Basophil', 'CA19-9', 'CEA', 'CPK', 'CRP', 'Creatinine', 'D.Bilirubin', 'Eosinophil', 'Fe', 'Free T4', 'GOT(AST)', 'GPT(ALT)', 'Globulin', 'HDL', 'Hct', 'Hgb', 'I.Bilirubin', 'LDH', 'LDL', 'Lymphocyte', 'MCH', 'MCHC', 'MCV', 'MPV', 'Monocyte', 'Neutroph', 'PCT', 'PDW', 'PH', 'Platelet', 'RBC', 'RDW', 'SG', 'T-Amylase', 'T.Bilirubin', 'T.Cholesterol', 'T.Protein', 'TIBC', 'TSH', 'Triglyceride', 'UIBC', 'Uric acid', 'WBC', 'current_glucose', 'e-GFR', 'future_glucose', 'glucose_change', 'r-GTP', '공복혈당', '나이', '당화혈색소', '맥박', '비만도', '시력(우안)', '시력(좌안)', '신장', '안압(우안)', '안압(좌안)', '청력우(1000Hz)', '청력좌(1000Hz)', '체중', '허리둘레', '혈압(수축기)', '혈압(이완기)', 'gender_여자', 'Bilirubin_음성', 'Blood_음성', 'Glucose_음성', 'Keton_음성', 'Leukocyte_음성', 'Nitrite_음성', 'Protein_음성', 'Urobilinogen_음성', 'HBs-Ag_음성', 'HBs-Ab_음성']

# ── 경로 ─────────────────────────────────────────────────────────────────────
OUTPUT_ROOT      = ROOT / 'outputs' / '260601_DL_hparam_search'
LOG_PATH         = OUTPUT_ROOT / 'search_log.csv'
DIABETES_DATA    = ROOT / 'outputs' / '260526_preprocessing'      / 'diabetes_dataset.xlsx'
PREDIABETES_DATA = ROOT / 'outputs' / '260526_preprocessing'      / 'pre_diabetes_dataset.xlsx'
DIABETES_VIF     = ROOT / 'outputs' / '260527_statistic_analysis' / 'diabetes'     / 'vif_survived.csv'
PREDIABETES_VIF  = ROOT / 'outputs' / '260527_statistic_analysis' / 'pre_diabetes' / 'vif_survived.csv'

for p in [OUTPUT_ROOT / 'diabetes', OUTPUT_ROOT / 'pre_diabetes']:
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
searcher = HParamSearcher(
    output_root=OUTPUT_ROOT,
    n_folds=N_FOLDS,
    max_epochs=MAX_EPOCHS,
    early_stopping=EARLY_STOPPING,
    patience=PATIENCE,
    loss=LOSS,
)

searcher.search(
    n_trials=N_TRIALS,
    param_grid=PARAM_GRID,
    log_path=LOG_PATH,
    diabetes_data=DIABETES_DATA,
    diabetes_vif=DIABETES_VIF,
    diabetes_features=DIABETES_FEATURES,
    prediabetes_data=PREDIABETES_DATA,
    prediabetes_vif=PREDIABETES_VIF,
    prediabetes_features=PREDIABETES_FEATURES,
)


이어서 탐색: 0번부터 시작
현재 최고 F1  |  Diabetes: -1.0000  |  Pre-diabetes: -1.0000


In [ ]:
searcher.show_best()


Diabetes  |  Mean F1 = 0.7141  |  Trial #4  |  2026-06-02T10:56:46
  random_state              = 99
  imputation_strategy       = mean
  mice_max_iter             = 10
  knn_n_neighbors           = 10
  imputation_fill_value     = -1.0
  hidden_layer_sizes        = [256, 128, 64, 32]
  dropout_rate              = 0.0
  learning_rate             = 0.005
  weight_decay              = 0.01
  batch_size                = 64
  validation_fraction       = 0.15



,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc,threshold,n_epochs
fold,,,,,,,,,
1,0.986755,1.000000,0.986111,0.777778,0.875000,1.000000,1.000000,0.606129,62
2,0.960265,0.857143,0.965278,0.545455,0.666667,0.987103,0.701020,0.193688,65
3,0.966887,0.375000,1.000000,1.000000,0.545455,0.972028,0.755801,0.934787,44
4,0.980000,0.714286,0.993007,0.833333,0.769231,0.872128,0.844197,0.120940,71



Pre-diabetes  |  Mean F1 = 0.8682  |  Trial #10  |  2026-06-02T10:59:30
  random_state              = 99
  imputation_strategy       = mean
  mice_max_iter             = 10
  knn_n_neighbors           = 20
  imputation_fill_value     = -1.0
  hidden_layer_sizes        = [256]
  dropout_rate              = 0.3
  learning_rate             = 0.0005
  weight_decay              = 0.01
  batch_size                = 8
  validation_fraction       = 0.25



,accuracy,sensitivity,specificity,precision,f1,auc,pr_auc,threshold,n_epochs
fold,,,,,,,,,
1,0.946429,0.933333,0.950192,0.843373,0.886076,0.965262,0.944044,0.585450,54
2,0.928358,0.770270,0.973180,0.890625,0.826087,0.966087,0.921054,0.639084,57
3,0.937313,0.837838,0.965517,0.873239,0.855172,0.972559,0.933532,0.632633,54
4,0.958209,0.905405,0.973180,0.905405,0.905405,0.980946,0.964431,0.676776,51


In [ ]:
searcher.show_top10(LOG_PATH)


총 10회 시도
── Diabetes Top-10 ──


,trial,imstrategy,layerssizes,learning_rate,dropout_rate,batch_size,validation_fraction,random_state,diabetes_f1,prediabetes_f1
3,4,mean,"(256, 128, 64, 32)",0.005000,0.000000,64,0.150000,99,0.7141,0.8170
9,10,mean,"(256,)",0.000500,0.300000,8,0.250000,99,0.7128,0.8682
5,6,most_frequent,"(256, 128, 64, 32)",0.003000,0.500000,128,0.300000,2024,0.6634,0.8340
4,5,most_frequent,"(128,)",0.000500,0.200000,16,0.250000,99,0.6609,0.8513
2,3,median,"(256, 128, 64, 32)",0.000300,0.400000,8,0.200000,1,0.5796,0.8633
8,9,knn,"(128,)",0.005000,0.100000,32,0.300000,42,0.5612,0.8290
1,2,median,"(128, 64, 32)",0.000500,0.200000,128,0.250000,42,0.4775,0.8056
6,7,knn,"(64,)",0.000100,0.000000,64,0.300000,2024,0.3933,0.8439
0,1,mice,"(64, 128)",0.000300,0.400000,8,0.200000,0,0.3872,0.8527
7,8,mean,"(256, 128, 64, 32)",0.000300,0.100000,32,0.150000,7,0.2889,0.8232



── Pre-diabetes Top-10 ──


,trial,imstrategy,layerssizes,learning_rate,dropout_rate,batch_size,validation_fraction,random_state,diabetes_f1,prediabetes_f1
9,10,mean,"(256,)",0.000500,0.300000,8,0.250000,99,0.7128,0.8682
2,3,median,"(256, 128, 64, 32)",0.000300,0.400000,8,0.200000,1,0.5796,0.8633
0,1,mice,"(64, 128)",0.000300,0.400000,8,0.200000,0,0.3872,0.8527
4,5,most_frequent,"(128,)",0.000500,0.200000,16,0.250000,99,0.6609,0.8513
6,7,knn,"(64,)",0.000100,0.000000,64,0.300000,2024,0.3933,0.8439
5,6,most_frequent,"(256, 128, 64, 32)",0.003000,0.500000,128,0.300000,2024,0.6634,0.8340
8,9,knn,"(128,)",0.005000,0.100000,32,0.300000,42,0.5612,0.8290
7,8,mean,"(256, 128, 64, 32)",0.000300,0.100000,32,0.150000,7,0.2889,0.8232
3,4,mean,"(256, 128, 64, 32)",0.005000,0.000000,64,0.150000,99,0.7141,0.8170
1,2,median,"(128, 64, 32)",0.000500,0.200000,128,0.250000,42,0.4775,0.8056
